In [11]:
import random
from matplotlib import pyplot as plt

import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.weightstats import ttest_ind

import helpers

### Data Loading

In [12]:
attribute_name = "katz"
attributes = helpers.load_attributes(attribute_name)
attribute2favorability = helpers.load_favorability_ratings()

variables = ["spa_Latn", "spanish"]
variable2type = {
    "spa_Latn": "covert",
    "spanish": "overt"
}

stereotype_types = ["overt", "covert"]


In [13]:
stereotype_results = pd.DataFrame()
for variable in variables:
    for model in helpers.MODELS:
        results = helpers.load_results(
            model, 
            variable, 
            attribute_name,
            True
        )
        results_df = helpers.results2df(
            results, 
            attributes, 
            model,
            variable,
            True
        )
        results_df["type"] = variable2type[variable]
        stereotype_results = pd.concat([
            stereotype_results, 
            results_df, 
        ])

### Adjective analysis

In [14]:
k = 5

for stereotype_type in stereotype_types:
    print(f"Language models ({stereotype_type})")
    for model in helpers.FAMILIES:
        attributes_model = stereotype_results[
            (stereotype_results.family==model) &
            (stereotype_results.type==stereotype_type)
        ].groupby("attribute", as_index=False)["ratio"].mean()
        attributes_model_ranked = attributes_model.sort_values( 
            by="ratio",
            ascending=False
        )["attribute"].tolist()
        print(
            model, 
            attributes_model_ranked[:k], 
            [attribute2favorability[a] for a in attributes_model_ranked[:k]]
        )

Language models (overt)
llama3 ['musical', 'artistic', 'passionate', 'aggressive', 'honest'] [1.08, 1.12, 1.02, -0.58, 1.58]
Language models (covert)
llama3 ['neat', 'loud', 'religious', 'progressive', 'rude'] [0.85, -0.65, 0.15, 0.8, -1.46]


### Agreement analysis

In [15]:
agreement_results = pd.DataFrame()
for stereotype_type in stereotype_types:
    agreement_list = []
    for model in helpers.FAMILIES:
        attributes_model = stereotype_results[
            (stereotype_results.family==model) &
            (stereotype_results.type==stereotype_type)
        ].groupby(["attribute", "prompt"], as_index=False)["ratio"].mean()
        for prompt in set(attributes_model.prompt):
            attributes_ranked = attributes_model[attributes_model.prompt==prompt].sort_values( 
                by="ratio",
                ascending=False
            )["attribute"].tolist()
    results_df = pd.DataFrame(
        agreement_list,
        columns=["ap", "family", "prompt", "experiment"]
    )
    results_df["type"] = stereotype_type
    agreement_results = pd.concat([
        agreement_results, 
        results_df, 
    ])